# **Generative Adversarial Network (GAN) on MNIST Handwriten Digits Generation**
---
---
AI & GenAI Assignment

**Objective:** To implement and train a simple Generative Adversarial Network (GAN) using TensorFlow
(Keras API) to generate handwritten digit images that resemble the MNIST dataset. <br>
**Component:** Generator, Disriminator (real/fake classifier) <br>
**Visualization:** Generated images grid , Loss curves

###1. Importing Libraries
---

In [1]:
import tensorflow as tf
from tensorflow.keras import layers

import numpy as np
import matplotlib.pyplot as plt
import os
import time
from IPython.display import clear_output

import warnings
warnings.filterwarnings('ignore')

##2. Data Loading & Preprocessing
---
- The MNIST dataset is loaded using TensorFlow.
- Images are normalized to [-1,1] because the generator uses Tanh activation.

In [2]:
# Load MNIST
#---------------
(x_train, _), (_, _) = tf.keras.datasets.mnist.load_data()
print(f"Training data shape:; {x_train.shape}")

Training data shape:; (60000, 28, 28)


In [3]:
# Normalize to [-1,1]
#------------------------------
x_train = x_train = (x_train - 127.5) / 127.5

In [4]:
# Reshape to (28,28,1)
#-----------------------
x_train = x_train.reshape(x_train.shape[0], 28, 28, 1).astype('float32')

# Reduce dataset size
x_train = x_train[:60000]

In [5]:


# Batch Dataset
#-----------------
BUFFER_SIZE = 60000
BATCH_SIZE = 128

train_dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(BUFFER_SIZE).batch(BATCH_SIZE,drop_remainder=True)

print("Training Data Shape:", x_train.shape)

Training Data Shape: (60000, 28, 28, 1)


##3. Model Definitions
---

###3.1 Generator Model
The Generator:

- Takes 100-d noise
- Upsamples to 28×28 image
- Final activation = Tanh

In [6]:
def build_generator(latent_dim=100):
    model = tf.keras.Sequential([
        # Input noise = Dense
        layers.Dense(7*7*256, use_bias=False, input_shape=(latent_dim,),
                     kernel_initializer='he_normal'),
        layers.BatchNormalization(),
        layers.LeakyReLU(alpha=0.2),

        # Reshape to feature map
        layers.Reshape((7,7,256)),

        # Upsample to 14 * 14
        layers.Conv2DTranspose(128, (5,5), strides=(2,2),
                               padding='same', use_bias=False,
                               kernel_initializer='he_normal'),
        layers.BatchNormalization(),
        layers.LeakyReLU(alpha=0.2),

        # Upsample to 28*28
        layers.Conv2DTranspose(64, (5,5), strides=(2,2),
                               padding='same', use_bias=False,
                               kernel_initializer='he_normal'),
        layers.BatchNormalization(),
        layers.LeakyReLU(alpha=0.2),

        # Output layer: 28*28*1, tanh activation for [-1,1]
        layers.Conv2DTranspose(1, (5,5), strides=(1,1),
                               padding='same', use_bias=False,
                               activation='tanh')
    ])
    return model

generator = build_generator()

generator.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 12544)          │     1,254,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 12544)          │        50,176 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 7, 7, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose                │ (None, 14, 14, 128)    │       819,200 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 14, 14, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_1              │ (None, 28, 28, 64)     │       204,800 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 28, 28, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 28, 28, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_2              │ (None, 28, 28, 1)      │         1,600 │
│ (Conv2DTranspose)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,330,944 (8.89 MB)

 Trainable params: 2,305,472 (8.79 MB)

 Non-trainable params: 25,472 (99.50 KB)

###3.2 Discriminator Model
---
The Discriminator:

- Classifies image as real or fake
- Uses Conv2D, Dropout, Dense (Sigmoid)

In [7]:
def build_discriminator():
    model = tf.keras.Sequential([
        layers.Conv2D(64, (5,5), strides=(2,2),
                      padding='same',
                      input_shape=[28,28,1],
                      kernel_initializer='he_normal'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(128, (5,5), strides=(2,2),
                      padding='same',
                      kernel_initializer='he_normal'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(256, (5,5), strides=(2,2),
                      padding='same',
                      kernel_initializer='he_normal'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

discriminator = build_discriminator()
discriminator.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 14, 14, 64)     │         1,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 7, 7, 128)      │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_4 (LeakyReLU)       │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 4, 4, 256)      │       819,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_5 (LeakyReLU)       │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │         4,097 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,030,145 (3.93 MB)

 Trainable params: 1,030,145 (3.93 MB)

 Non-trainable params: 0 (0.00 B)

##4. Loss Functions & Optimizers
---------


In [8]:
cross_entropy = tf.keras.losses.BinaryCrossentropy()

generator_optimizer = tf.keras.optimizers.Adam(0.0002, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(0.0002, beta_1=0.5)

# Label smoothing used
def discriminator_loss(real_output, fake_output):
    real_labels = tf.ones_like(real_output) * 0.9
    fake_labels = tf.zeros_like(fake_output)
    real_loss = cross_entropy(real_labels, real_output)
    fake_loss = cross_entropy(fake_labels, fake_output)
    return real_loss + fake_loss

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output) * 0.9, fake_output)

##5. Training Loop
---
The training loop:

- Generates fake images
- Computes losses
- Applies gradients
- Saves losses for plotting
-






Displays generated images

In [9]:
EPOCHS = 20
noise_dim = 100
num_examples_to_generate = 16

seed = tf.random.normal([num_examples_to_generate, noise_dim])

generator_losses = []
discriminator_losses = []

@tf.function
def train_step(images):
    noise = tf.random.normal([BATCH_SIZE, noise_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    # Compute gradients
    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    # Apply gradients
    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss


def train(dataset, epochs):
    for epoch in range(epochs):
        gen_loss_epoch = 0
        disc_loss_epoch = 0

        for image_batch in dataset:
            gen_loss, disc_loss = train_step(image_batch)
            gen_loss_epoch += gen_loss
            disc_loss_epoch += disc_loss

        generator_losses.append(gen_loss_epoch.numpy())
        discriminator_losses.append(disc_loss_epoch.numpy())

        print(f'Epoch {epoch+1}, Gen Loss: {gen_loss_epoch:.4f}, Disc Loss: {disc_loss_epoch:.4f}')

        generate_images(generator, epoch+1, seed)

##6. Sample Image Visualization
---

In [10]:
#def generate_images(model, epoch, test_input):
    #predictions = model(test_input, training=False)

    #plt.figure(figsize=(4,4))

    #for i in range(predictions.shape[0]):
        #plt.subplot(4,4,i+1)
        #plt.imshow(predictions[i,:,:,0] * 127.5 + 127.5, cmap='gray')
        #plt.axis('off')

    #plt.suptitle(f'Epoch {epoch}')
    #plt.show()

In [11]:
def generate_and_save_images(model, epoch, test_input):
    predictions = model(test_input, training=False)

    plt.figure(figsize=(6,6))

    for i in range(16):
        plt.subplot(4,4,i+1)
        plt.imshow(predictions[i,:,:,0] * 127.5 + 127.5, cmap='gray')
        plt.axis('off')

    plt.suptitle(f'Generated Images After {epoch} Epochs')
    plt.tight_layout()
    plt.savefig("Final_Generated_Digits.png", dpi=300)
    plt.show()

In [15]:
def train(dataset, epochs):
    for epoch in range(epochs):
        start = time.time()

        epoch_gen_loss = []
        epoch_disc_loss = []

        for image_batch in dataset:
            gen_loss, disc_loss = train_step(image_batch)
            epoch_gen_loss.append(gen_loss)
            epoch_disc_loss.append(disc_loss)

        generator_losses.append(
            tf.reduce_mean(epoch_gen_loss).numpy()
        )
        discriminator_losses.append(
            tf.reduce_mean(epoch_disc_loss).numpy()
        )

        print(
            f"Epoch {epoch+1}, "
            f"Gen Loss: {generator_losses[-1]:.4f}, "
            f"Disc Loss: {discriminator_losses[-1]:.4f}, "
            f"Time: {time.time()-start:.2f}s"
        )

    generate_and_save_images(generator, epochs, seed)

In [ ]:
train(train_dataset, EPOCHS)

##7. Plot Loss Curves
---



In [ ]:
plt.figure(figsize=(8,5))
plt.plot(generator_losses, label='Generator Loss')
plt.plot(discriminator_losses, label='Discriminator Loss')
plt.legend()
plt.title("GAN Training Loss")
plt.show()